# Notebook 02: Hypothesis Testing and Experiment Evaluation

> **Tutorial question:** Do the emails change customer behavior, and which campaign should the business choose?

Notebook 01 established that the randomized design is credible. We now estimate treatment effects for three outcomes and three pairwise comparisons. The analysis uses a two-proportion z-test for binary outcomes and Welch's t-test for spend.

### What you will learn
- How to translate a 0/1 outcome into a difference in rates.
- Why the test statistic and confidence interval answer different but complementary questions.
- Why multiple-comparisons correction matters when nine hypotheses are tested.
- How to separate statistical significance from practical business impact.

### Decision rule
We report the estimated lift, its 95% confidence interval, and the Benjamini-Hochberg-adjusted p-value. A recommendation should consider both uncertainty and lift size, not p-value alone.

In [1]:
import pandas as pd
import numpy as np

from statsmodels.stats.proportion import proportions_ztest
from scipy.stats import ttest_ind
from statsmodels.stats import multitest

In [2]:
hillstrom_df = pd.read_csv("../data/raw/hillstrom.csv")

display(hillstrom_df.head(), hillstrom_df.tail())

,recency,history_segment,history,mens,womens,zip_code,newbie,channel,segment,visit,conversion,spend
0,10,2) $100 - $200,142.44,1,0,Surburban,0,Phone,Womens E-Mail,0,0,0.0
1,6,3) $200 - $350,329.08,1,1,Rural,1,Web,No E-Mail,0,0,0.0
2,7,2) $100 - $200,180.65,0,1,Surburban,1,Web,Womens E-Mail,0,0,0.0
3,9,5) $500 - $750,675.83,1,0,Rural,1,Web,Mens E-Mail,0,0,0.0
4,2,1) $0 - $100,45.34,1,0,Urban,0,Web,Womens E-Mail,0,0,0.0


,recency,history_segment,history,mens,womens,zip_code,newbie,channel,segment,visit,conversion,spend
63995,10,2) $100 - $200,105.54,1,0,Urban,0,Web,Mens E-Mail,0,0,0.0
63996,5,1) $0 - $100,38.91,0,1,Urban,1,Phone,Mens E-Mail,0,0,0.0
63997,6,1) $0 - $100,29.99,1,0,Urban,1,Phone,Mens E-Mail,0,0,0.0
63998,1,5) $500 - $750,552.94,1,0,Surburban,1,Multichannel,Womens E-Mail,0,0,0.0
63999,1,4) $350 - $500,472.82,0,1,Surburban,0,Web,Mens E-Mail,0,0,0.0


## Hypothesis testing for impact metrics

For `visit` and `conversion`, `sum` is the number of successes and `count` is the number of customers. The estimated effect is `p1 - p2`, so the sign follows the order in `comparisons`. For `spend`, the effect is the difference in mean dollars per customer.

In [3]:
# Visits

comparisons = [("Mens E-Mail", "No E-Mail"), ("Mens E-Mail", "Womens E-Mail"), ("Womens E-Mail", "No E-Mail")]

visit_counts_df = hillstrom_df.groupby("segment")["visit"].agg(["sum", "count"])

# TODO: For every pair in `comparisons`, run a two-proportion z-test.
# Calculate p1 - p2 and a 95% confidence interval using the unpooled
# standard error. Store the result in `visit_df` and add `Metric = "Visit"`.
# Required columns: Group 1, Group 2, Z-Statistic, P-Value, Difference,
# 95% CI, Metric.
res = []
for group1, group2 in comparisons:
    sum1 = visit_counts_df.loc[group1, "sum"]
    n1 = visit_counts_df.loc[group1, "count"]
    sum2 = visit_counts_df.loc[group2, "sum"]
    n2 = visit_counts_df.loc[group2, "count"]

    z_stat, p_val = proportions_ztest([sum1, sum2], [n1, n2])
    p1, p2 = sum1 / n1, sum2 / n2
    diff = p1 - p2
    se = np.sqrt(p1 * (1 - p1) / n1 + p2 * (1 - p2) / n2)
    ci_lower = round(diff - 1.96 * se, 4)
    ci_upper = round(diff + 1.96 * se, 4)
    ci = [ci_lower, ci_upper]
    res.append((group1, group2, z_stat, p_val, diff, ci))
visit_df = pd.DataFrame(
    res,
    columns=[
        "Group 1",
        "Group 2",
        "Z-Statistic",
        "P-Value",
        "Difference",
        "95% CI"
    ]
)
visit_df["Metric"] = "Visit"
visit_df


,Group 1,Group 2,Z-Statistic,P-Value,Difference,95% CI,Metric
0,Mens E-Mail,No E-Mail,22.486041,5.685165e-112,0.076590,"[0.07, 0.0832]",Visit
1,Mens E-Mail,Womens E-Mail,8.684558,3.802165e-18,0.031356,"[0.0243, 0.0384]",Visit
2,Womens E-Mail,No E-Mail,13.949181,3.182403e-44,0.045233,"[0.0389, 0.0516]",Visit


In [4]:
# Conversions

conversion_counts_df = hillstrom_df.groupby("segment")["conversion"].agg(["sum", "count"])

# TODO: Repeat the two-proportion z-test analysis for `conversion`.
# Calculate p1 - p2, the 95% CI, and store the table as `conversion_df`
# with the same columns as `visit_df`. Add `Metric = "Conversion"`.
conversion_counts_df
res = []
for group1, group2 in comparisons:
    sum1 = conversion_counts_df.loc[group1, "sum"]
    n1 = conversion_counts_df.loc[group1, "count"]
    sum2 = conversion_counts_df.loc[group2, "sum"]
    n2 = conversion_counts_df.loc[group2, "count"]

    z_stat, p_val = proportions_ztest([sum1, sum2], [n1, n2])
    p1, p2 = sum1 / n1, sum2 / n2
    diff = p1 - p2
    se = np.sqrt(p1 * (1 - p1) / n1 + p2 * (1 - p2) / n2)
    ci_lower = round(diff - 1.96 * se, 4)
    ci_upper = round(diff + 1.96 * se, 4)
    ci = [ci_lower, ci_upper]
    res.append((group1, group2, z_stat, p_val, diff, ci))
conversion_df = pd.DataFrame(
    res,
    columns=[
        "Group 1",
        "Group 2",
        "Z-Statistic",
        "P-Value",
        "Difference",
        "95% CI"
    ]
)
conversion_df["Metric"] = "Conversion"
conversion_df

,Group 1,Group 2,Z-Statistic,P-Value,Difference,95% CI,Metric
0,Mens E-Mail,No E-Mail,7.385114,1.523224e-13,0.006805,"[0.005, 0.0086]",Conversion
1,Mens E-Mail,Womens E-Mail,3.712584,2.051538e-04,0.003694,"[0.0017, 0.0056]",Conversion
2,Womens E-Mail,No E-Mail,3.779561,1.571051e-04,0.003111,"[0.0015, 0.0047]",Conversion


In [ ]:
# Spend

# TODO: For every pair, run Welch's t-test (`equal_var=False`).
# Calculate the mean difference, Welch-style standard error, 95% CI,
# and Cohen's d. Store the table as `spend_df` and add `Metric = "Spend"`.
res = []
spend_count_df = hillstrom_df.groupby("segment")["spend"]
for group1, group2 in comparisons:
    group1_data = spend_count_df.get_group(group1)
    group2_data = spend_count_df.get_group(group2)
    t_stat, p_val = ttest_ind(group1_data, group2_data, equal_var=False)

    mean_diff = group1_data.mean() - group2_data.mean()
    se = np.sqrt(group1_data.var(ddof=1) / len(group1_data) + group2_data.var(ddof=1) / len(group2_data))
    ci_lower = round(mean_diff - 1.96 * se, 4)
    ci_upper = round(mean_diff + 1.96 * se, 4)
    cohen_d = mean_diff / np.sqrt((group1_data.var(ddof=1) + group2_data.var(ddof=1)) / 2)
    res.append((group1, group2, t_stat, p_val, mean_diff, [ci_lower, ci_upper], cohen_d))
spend_df = pd.DataFrame(
    res,
    columns=[
        "Group 1",
        "Group 2",
        "T-Statistic",
        "P-Value",
        "Difference",
        "95% CI",
        "Cohen's d"
    ]
)
spend_df["Metric"] = "Spend"
spend_df



,Group 1,Group 2,T-Statistic,P-Value,Difference,95% CI,Cohen's d,Metric
0,Mens E-Mail,No E-Mail,5.300140,1.163815e-07,0.769827,"[0.4851, 1.0545]",0.051350,Spend
1,Mens E-Mail,Womens E-Mail,2.164018,3.046865e-02,0.345415,"[0.0326, 0.6583]",0.020949,Spend
2,Womens E-Mail,No E-Mail,3.256372,1.129397e-03,0.424412,"[0.169, 0.6799]",0.031512,Spend


### Correct for multiple testing

There are 3 outcomes × 3 pairwise comparisons = 9 tests. Benjamini-Hochberg controls the expected false-discovery rate across this family of results. We use the adjusted p-values for the final significance call.

In [ ]:
# TODO: Rename the statistic columns consistently, concatenate all three
# result tables, and apply Benjamini-Hochberg to all nine raw p-values
# together with `multitest.multipletests(..., method="fdr_bh")`.
# Add `Reject H0` based on adjusted p < 0.05. Keep numeric p-values until
# the final display and store the combined table as `results_df`.
visit_df = visit_df.rename(
    columns={"Z-Statistic": "Test Statistic"}
)

conversion_df = conversion_df.rename(
    columns={"Z-Statistic": "Test Statistic"}
)

spend_df = spend_df.rename(
    columns={"T-Statistic": "Test Statistic"}
)

results_df = pd.concat(
    [visit_df, conversion_df, spend_df],
    ignore_index=True
)

reject, adjusted_pvalues, _, _ = multitest.multipletests(
    results_df["P-Value"],
    method="fdr_bh"
)

results_df["Adjusted P-Value"] = adjusted_pvalues
results_df["Reject H0"] = reject

results_df = results_df[
    [
        "Metric",
        "Group 1",
        "Group 2",
        "P-Value",
        "Adjusted P-Value",
        "Reject H0",
        "Difference",
        "95% CI",
        "Test Statistic",
        "Cohen's d",
    ]
]

resut

    



,Metric,Group 1,Group 2,P-Value,Adjusted P-Value,Reject H0,Difference,95% CI,Test Statistic,Cohen's d
0,Visit,Mens E-Mail,No E-Mail,5.685165e-112,5.116649e-111,True,0.076590,"[0.07, 0.0832]",22.486041,NaN
1,Visit,Mens E-Mail,Womens E-Mail,3.802165e-18,1.140650e-17,True,0.031356,"[0.0243, 0.0384]",8.684558,NaN
2,Visit,Womens E-Mail,No E-Mail,3.182403e-44,1.432081e-43,True,0.045233,"[0.0389, 0.0516]",13.949181,NaN
3,Conversion,Mens E-Mail,No E-Mail,1.523224e-13,3.427253e-13,True,0.006805,"[0.005, 0.0086]",7.385114,NaN
4,Conversion,Mens E-Mail,Womens E-Mail,2.051538e-04,2.637691e-04,True,0.003694,"[0.0017, 0.0056]",3.712584,NaN
5,Conversion,Womens E-Mail,No E-Mail,1.571051e-04,2.356577e-04,True,0.003111,"[0.0015, 0.0047]",3.779561,NaN
6,Spend,Mens E-Mail,No E-Mail,1.163815e-07,2.094867e-07,True,0.769827,"[0.4851, 1.0545]",5.300140,0.051350
7,Spend,Mens E-Mail,Womens E-Mail,3.046865e-02,3.046865e-02,True,0.345415,"[0.0326, 0.6583]",2.164018,0.020949
8,Spend,Womens E-Mail,No E-Mail,1.129397e-03,1.270572e-03,True,0.424412,"[0.169, 0.6799]",3.256372,0.031512


### Sensitivity check: winsorize extreme spend

Because spend has a long right tail, we cap observations at the 99.9th percentile and rerun the spend comparisons. If the business conclusion changes dramatically, the result is sensitive to a few large purchases; if it does not, confidence in the conclusion improves.

In [8]:
# TODO: Inspect upper quantiles of `spend` and choose the 99.9th
# percentile as the winsorization cap. Store it as `cap_value`.
cap_value = hillstrom_df["spend"].quantile(0.999)


In [11]:
# TODO: Clip spend in each arm at `cap_value`, rerun Welch's t-test for
# all three comparisons, and calculate the difference plus 95% CI.
# Save the numeric result as `winsorized_df` and compare it with `spend_df`.
# 创建 Winsorized 数据集
winsorized_data = hillstrom_df.copy()

winsorized_data["spend"] = winsorized_data["spend"].clip(
    upper=cap_value
)
spend_count_df_win = winsorized_data.groupby("segment")["spend"]
res = []
for group1, group2 in comparisons:
    group1_data = spend_count_df_win.get_group(group1)
    group2_data = spend_count_df_win.get_group(group2)
    t_stat, p_val = ttest_ind(group1_data, group2_data, equal_var=False)

    mean_diff = group1_data.mean() - group2_data.mean()
    se = np.sqrt(group1_data.var(ddof=1) / len(group1_data) + group2_data.var(ddof=1) / len(group2_data))
    ci_lower = round(mean_diff - 1.96 * se, 4)
    ci_upper = round(mean_diff + 1.96 * se, 4)
    cohen_d = mean_diff / np.sqrt((group1_data.var(ddof=1) + group2_data.var(ddof=1)) / 2)
    res.append((group1, group2, t_stat, p_val, mean_diff, [ci_lower, ci_upper], cohen_d))
spend_df_win = pd.DataFrame(
    res,
    columns=[
        "Group 1",
        "Group 2",
        "T-Statistic",
        "P-Value",
        "Difference",
        "95% CI",
        "Cohen's d"
    ]
)
spend_df_win["Metric"] = "Spend"
spend_df_win




,Group 1,Group 2,T-Statistic,P-Value,Difference,95% CI,Cohen's d,Metric
0,Mens E-Mail,No E-Mail,5.758189,8.569090e-09,0.659158,"[0.4348, 0.8835]",0.055788,Spend
1,Mens E-Mail,Womens E-Mail,2.167190,3.022593e-02,0.276241,"[0.0264, 0.5261]",0.020979,Spend
2,Womens E-Mail,No E-Mail,3.620066,2.948949e-04,0.382917,"[0.1756, 0.5902]",0.035031,Spend


## Translate the effect into business units

A proportion difference such as 0.0766 is 7.66 percentage points, not 0.0766 percent. Multiplying the per-customer effect and its confidence interval by 10,000 turns the statistical result into an operational planning number.

In [12]:
# TODO: Filter `results_df` to Men's Email vs No Email. Multiply the
# estimated difference and both 95% CI bounds by 10,000 to report
# incremental visits, conversions, or dollars. Display `impact_df`.
impact_df = results_df[
    (results_df["Group 1"] == "Mens E-Mail") & (results_df["Group 2"] == "No E-Mail")
].copy()
impact_df["Difference"] = impact_df["Difference"] * 10000
impact_df["95% CI"] = impact_df["95% CI"].apply(lambda x: [bound * 10000 for bound in x])
impact_df


,Metric,Group 1,Group 2,P-Value,Adjusted P-Value,Reject H0,Difference,95% CI,Test Statistic,Cohen's d
0,Visit,Mens E-Mail,No E-Mail,5.685165e-112,5.116649e-111,True,765.895637,"[700.0000000000001, 832.0]",22.486041,NaN
3,Conversion,Mens E-Mail,No E-Mail,1.523224e-13,3.427253e-13,True,68.050065,"[50.0, 86.0]",7.385114,NaN
6,Spend,Mens E-Mail,No E-Mail,1.163815e-07,2.094867e-07,True,7698.271559,"[4851.0, 10545.0]",5.300140,0.05135


## Summary and campaign recommendation

The three pairwise comparisons were evaluated for visit, conversion, and spend. After applying the Benjamini-Hochberg correction across all nine tests, every comparison remained statistically significant (adjusted p < 0.05). All estimated effects are positive, so both email campaigns improved outcomes relative to the comparison groups.

- **Visit:** Men's Email increased the visit rate by 7.66 percentage points versus No Email (95% CI: 7.00 to 8.32 percentage points), and by 3.14 percentage points versus Women's Email (95% CI: 2.43 to 3.84). Women's Email increased visits by 4.52 percentage points versus No Email (95% CI: 3.89 to 5.16).
- **Conversion:** Men's Email increased conversion by 0.68 percentage points versus No Email (95% CI: 0.50 to 0.86), and by 0.37 percentage points versus Women's Email (95% CI: 0.17 to 0.56). Women's Email increased conversion by 0.31 percentage points versus No Email (95% CI: 0.15 to 0.47).
- **Spend:** Men's Email increased mean spend by $0.77 per customer versus No Email (95% CI: $0.49 to $1.05), and by $0.35 versus Women's Email (95% CI: $0.03 to $0.66). Women's Email increased mean spend by $0.42 versus No Email (95% CI: $0.17 to $0.68). Cohen's d values were small (0.021 to 0.051), so the effects are statistically reliable but modest relative to customer-level spend variation.

The 99.9th-percentile Winsorization sensitivity check reduced the spend differences, but did not change their direction or individual significance: the Men's-versus-No-Email difference became $0.66 (95% CI: $0.43 to $0.88), while the Men's-versus-Women's difference became $0.28 (95% CI: $0.03 to $0.53). This suggests that extreme purchases contribute to the magnitude of the raw estimate, but do not explain the overall conclusion.

For every 10,000 customers, Men's Email is estimated to generate approximately **766 additional visits** (95% CI: 700 to 832), **68 additional conversions** (95% CI: 50 to 86), and **$7,698 additional customer spend** (95% CI: $4,851 to $10,545) relative to No Email.

**Recommendation:** Men's Email is the strongest campaign in this experiment because it outperformed No Email and Women's Email on all three outcomes. The rollout decision should still compare the incremental customer spend with campaign delivery costs and margin; statistical significance alone does not guarantee positive profit.